In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from IPython.display import display
import sys
sns.set_theme(style="whitegrid")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))


from utils.data_utils import generate_rich_link_dataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
def chronological_split_and_clean(df, train_ratio=0.8):
    # Ordinamento cronologico
    df_sorted = df.sort_values('planning_date').reset_index(drop=True)
    split_idx = int(len(df_sorted) * train_ratio)
    
    df_train = df_sorted.iloc[:split_idx].copy()
    df_test = df_sorted.iloc[split_idx:].copy()
    
    # Rimozione date (non più necessarie dopo lo split)
    df_train = df_train.drop(columns=['planning_date'])
    df_test = df_test.drop(columns=['planning_date'])
    
    # Ricerca feature costanti sul train set
    constant_cols = [col for col in df_train.columns if df_train[col].nunique() <= 1 and col != 'link_label']
    if constant_cols:
        df_train = df_train.drop(columns=constant_cols)
        df_test = df_test.drop(columns=constant_cols, errors='ignore')
        print(f"Rimosse feature costanti: {constant_cols}")
        
    print(f"Train set: {len(df_train)} record.")
    print(f"Test set:  {len(df_test)} record.")
    
    return df_train, df_test

In [9]:
def process_correlation(df_train, df_test, corr_thresh=0.90):
    cols_to_exclude = ['patient_id', 'operator_id', 'link_label']
    features_for_corr = df_train.drop(columns=cols_to_exclude, errors='ignore').select_dtypes(include=[np.number])
    corr_matrix = features_for_corr.corr().abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > corr_thresh)]
    
    if to_drop:
        df_train = df_train.drop(columns=to_drop)
        df_test = df_test.drop(columns=to_drop)
        print(f"Feature scartate per multicollinearità (> {corr_thresh}): {to_drop}")
    else:
        print("Nessuna feature scartata per multicollinearità.")
        
    return df_train, df_test

In [10]:
def final_encoding_and_export(df_train, df_test, org_name, cat_thresh=0.05, output_dir="data"):
    os.makedirs(output_dir, exist_ok=True)
    df_tr = df_train.copy()
    df_te = df_test.copy()

    # 1. Raggruppamento categorie rare
    cat_cols = df_tr.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
    for col in cat_cols:
        freq = df_tr[col].value_counts(normalize=True)
        frequent_cats = freq[freq >= cat_thresh].index.tolist()
        df_tr[col] = df_tr[col].where(df_tr[col].isin(frequent_cats), 'Other')
        df_te[col] = df_te[col].where(df_te[col].isin(frequent_cats), 'Other')

    # 2. One-Hot Encoding
    cols_to_encode = [c for c in cat_cols if c not in ['operator_id', 'patient_id']]
    df_tr_encoded = pd.get_dummies(df_tr, columns=cols_to_encode, drop_first=True, dtype=int)
    df_te_encoded = pd.get_dummies(df_te, columns=cols_to_encode, drop_first=True, dtype=int)

    # 3. Allineamento Colonne
    df_tr_encoded, df_te_encoded = df_tr_encoded.align(df_te_encoded, join='left', axis=1, fill_value=0)

    # 4. Scaling delle feature continue
    cols_to_exclude = ['patient_id', 'operator_id', 'link_label']
    num_cols = [c for c in df_tr_encoded.columns if c not in cols_to_exclude and df_tr_encoded[c].nunique() > 2]
    
    scaler = StandardScaler()
    df_tr_encoded[num_cols] = scaler.fit_transform(df_tr_encoded[num_cols])
    df_te_encoded[num_cols] = scaler.transform(df_te_encoded[num_cols])

    # 5. Esportazione
    train_path = f"{output_dir}/train_dataset_{org_name}.csv"
    test_path = f"{output_dir}/test_dataset_{org_name}.csv"
    df_tr_encoded.to_csv(train_path, index=False)
    df_te_encoded.to_csv(test_path, index=False)
    
    print(f"   - Train CSV salvato in: {train_path} ({df_tr_encoded.shape[1]} feature)")
    print(f"   - Test CSV salvato in:  {test_path} ({df_te_encoded.shape[1]} feature)")

    return df_tr_encoded, df_te_encoded

In [11]:
def generate_statistical_report(df, title="Report Statistico"):
    """
    Genera un DataFrame riassuntivo con tipi di dato, valori mancanti e distribuzioni.
    """
    print(f"\n--- {title} ---")
    report_data = []
    
    for col in df.columns:
        n_missing = df[col].isnull().sum()
        pct_missing = (n_missing / len(df)) * 100
        n_unique = df[col].nunique()
        dtype = str(df[col].dtype)
        
        # Logica di visualizzazione della distribuzione
        if pd.api.types.is_numeric_dtype(df[col]) and n_unique > 2:
            dist_info = f"Min: {df[col].min():.1f} | Media: {df[col].mean():.1f} | Max: {df[col].max():.1f}"
        else:
            # Per le variabili categoriche o booleane
            top_vals = df[col].value_counts().head(3).to_dict()
            dist_info = "Top: " + ", ".join([f"{k} ({v})" for k, v in top_vals.items()])
            
        report_data.append({
            'Feature': col,
            'Tipo': dtype,
            'Distinti': n_unique,
            'Mancanti': f"{n_missing} ({pct_missing:.1f}%)",
            'Distribuzione': dist_info
        })
        
    report_df = pd.DataFrame(report_data)
    return report_df

In [12]:
all_json_files = glob.glob("data/*.json")

orgs_files = {
    "Org6": [f for f in all_json_files if "Org6" in f],
    "Org9": [f for f in all_json_files if "Org9" in f]
}

for org_name, files in orgs_files.items():
    if not files:
        print(f"Nessun file trovato per {org_name}.")
        continue
        
    df_base = generate_rich_link_dataset(files, negative_ratio=2, random_seed=42)
    df_train, df_test = chronological_split_and_clean(df_base)
    df_train_corr, df_test_corr = process_correlation(df_train, df_test)
    df_train_final, df_test_final = final_encoding_and_export(df_train_corr, df_test_corr, org_name)

    stats_base = generate_statistical_report(df_base, "Situazione Iniziale")
    display(stats_base)
    
    stats_final = generate_statistical_report(df_train_final, "Situazione Finale")
    display(stats_final)

Train set: 16284 record.
Test set:  4071 record.
Nessuna feature scartata per multicollinearità.
   - Train CSV salvato in: data/train_dataset_Org6.csv (17 feature)
   - Test CSV salvato in:  data/test_dataset_Org6.csv (17 feature)

--- Situazione Iniziale ---


,Feature,Tipo,Distinti,Mancanti,Distribuzione
0,planning_date,datetime64[us],63,0 (0.0%),"Top: 2023-01-31 00:00:00 (344), 2022-11-11 00:..."
1,patient_id,int64,434,0 (0.0%),Min: 16235.0 | Media: 32852.9 | Max: 39219.0
2,operator_id,int64,24,0 (0.0%),Min: 210.0 | Media: 246.1 | Max: 524.0
3,pat_type,str,5,0 (0.0%),"Top: O (8904), MAC8 (7732), N (3014)"
4,pat_needs_lifter,int64,2,0 (0.0%),"Top: 0 (19018), 1 (1337)"
5,sess_type,object,2,0 (0.0%),"Top: 0 (20277), 0 (78)"
6,sess_length,int64,4,0 (0.0%),Min: 30.0 | Media: 40.5 | Max: 70.0
7,sess_location,int64,2,0 (0.0%),"Top: 1 (20046), 2 (309)"
8,op_burdenScore,int64,5,0 (0.0%),Min: 2.0 | Media: 3.9 | Max: 7.0
9,op_effectiveTime,int64,7,0 (0.0%),Min: 210.0 | Media: 386.9 | Max: 480.0



--- Situazione Finale ---


,Feature,Tipo,Distinti,Mancanti,Distribuzione
0,patient_id,int64,370,0 (0.0%),Min: 16235.0 | Media: 31960.1 | Max: 37288.0
1,operator_id,int64,24,0 (0.0%),Min: 210.0 | Media: 245.8 | Max: 524.0
2,pat_needs_lifter,int64,2,0 (0.0%),"Top: 0 (15345), 1 (939)"
3,sess_length,float64,4,0 (0.0%),Min: -0.7 | Media: 0.0 | Max: 2.1
4,sess_location,int64,2,0 (0.0%),"Top: 1 (16029), 2 (255)"
5,op_burdenScore,float64,5,0 (0.0%),Min: -1.5 | Media: 0.0 | Max: 2.6
6,op_effectiveTime,float64,7,0 (0.0%),Min: -2.1 | Media: 0.0 | Max: 1.1
7,op_qual_count,float64,3,0 (0.0%),Min: -2.6 | Media: -0.0 | Max: 2.7
8,op_is_morning,int64,2,0 (0.0%),"Top: 1 (15688), 0 (596)"
9,op_is_afternoon,int64,2,0 (0.0%),"Top: 1 (15133), 0 (1151)"


Train set: 7942 record.
Test set:  1986 record.
Nessuna feature scartata per multicollinearità.
   - Train CSV salvato in: data/train_dataset_Org9.csv (17 feature)
   - Test CSV salvato in:  data/test_dataset_Org9.csv (17 feature)

--- Situazione Iniziale ---


,Feature,Tipo,Distinti,Mancanti,Distribuzione
0,planning_date,datetime64[us],62,0 (0.0%),"Top: 2023-01-24 00:00:00 (184), 2022-11-09 00:..."
1,patient_id,int64,257,0 (0.0%),Min: 24868.0 | Media: 33223.2 | Max: 39259.0
2,operator_id,int64,21,0 (0.0%),Min: 29.0 | Media: 119.8 | Max: 614.0
3,pat_type,str,3,0 (0.0%),"Top: N (5535), O (3937), CP (456)"
4,pat_needs_lifter,int64,2,0 (0.0%),"Top: 0 (9532), 1 (396)"
5,sess_type,object,3,0 (0.0%),"Top: 0 (8996), 1 (921), 0 (11)"
6,sess_length,int64,3,0 (0.0%),Min: 20.0 | Media: 26.7 | Max: 60.0
7,sess_location,int64,2,0 (0.0%),"Top: 1 (9559), 2 (369)"
8,op_burdenScore,int64,8,0 (0.0%),Min: 2.0 | Media: 4.2 | Max: 11.0
9,op_effectiveTime,int64,13,0 (0.0%),Min: 140.0 | Media: 262.9 | Max: 350.0



--- Situazione Finale ---


,Feature,Tipo,Distinti,Mancanti,Distribuzione
0,patient_id,int64,219,0 (0.0%),Min: 24868.0 | Media: 32391.9 | Max: 37532.0
1,operator_id,int64,21,0 (0.0%),Min: 29.0 | Media: 121.0 | Max: 614.0
2,pat_needs_lifter,int64,2,0 (0.0%),"Top: 0 (7710), 1 (232)"
3,sess_length,float64,3,0 (0.0%),Min: -1.3 | Media: -0.0 | Max: 6.8
4,sess_location,int64,2,0 (0.0%),"Top: 1 (7585), 2 (357)"
5,op_burdenScore,float64,8,0 (0.0%),Min: -1.6 | Media: -0.0 | Max: 5.2
6,op_effectiveTime,float64,13,0 (0.0%),Min: -1.7 | Media: -0.0 | Max: 1.1
7,op_qual_count,float64,3,0 (0.0%),Min: -1.1 | Media: -0.0 | Max: 1.7
8,op_is_morning,int64,2,0 (0.0%),"Top: 1 (6736), 0 (1206)"
9,op_is_afternoon,int64,2,0 (0.0%),"Top: 1 (6141), 0 (1801)"


In [13]:
def plot_features(df, features_to_plot=None, title_suffix="", save_dir="images"):
    """
    Genera grafici per le feature indicate e li salva nella directory specificata.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Se non specifichiamo le feature, le prende tutte tranne l'ID e la data
    cols = features_to_plot if features_to_plot is not None else [c for c in df.columns if c not in ['operator_id', 'planning_date']]
    
    for col in cols:
        if col not in df.columns:
            continue
            
        col_data = df[col].dropna()
        if len(col_data) == 0:
            continue
            
        plt.figure(figsize=(8, 5))
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        is_discrete = df[col].nunique() < 10  # Soglia per trattarla come categorica visiva
        
        plot_title = f"{col} {title_suffix}".strip()
        
        if is_numeric and not is_discrete:
            # Istogramma continuo (es. density_ratio)
            sns.histplot(col_data, kde=True, color="skyblue", bins=20)
            plt.axvline(col_data.mean(), color='red', linestyle='--', label=f'Media: {col_data.mean():.1f}')
            plt.ylabel('Frequenza (Giornate-Operatore)')
            plt.legend()
        else:
            # Grafico a barre
            ax = sns.countplot(data=df, x=col, palette="Set2", order=df[col].value_counts().index)
            plt.xticks(rotation=45)
            plt.ylabel('Conteggio')
            
            # Etichette sopra le barre
            for p in ax.patches:
                height = int(p.get_height())
                if height > 0:
                    ax.annotate(f'{height}', 
                                (p.get_x() + p.get_width() / 2., height), 
                                ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')
                    
        plt.title(plot_title, fontsize=14, pad=15)
        plt.xlabel(col)
        plt.tight_layout()
        
        file_suffix = f"_{title_suffix.strip().replace(' ', '_')}" if title_suffix else ""
        plt.savefig(f"{save_dir}/{col}{file_suffix}.png", bbox_inches='tight')
        plt.show()

# print("GENERAZIONE GRAFICI: PRIMA")
# plot_features(df_base, features_to_plot=["pat_type"], title_suffix="(Pre-Pulizia)")

# features_chiave_modificate = ['target_assignments', 'density_ratio'] 
# print("GENERAZIONE GRAFICI: DOPO")
# plot_features(df_train_final, features_to_plot=features_chiave_modificate, title_suffix="(Post-Pulizia)")